In [4]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict
import pandas as pd
import random
import os

seed = 3888
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

img_width, img_height = 100, 100
batch_size = 512
AUTOTUNE = tf.data.AUTOTUNE

tumour_folders = ["Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
immune_folders = ["CD4+_T_Cells", "CD8+_T_Cells", "B_Cells", "Stromal"]
sub_label_list = tumour_folders + immune_folders
sub_label_to_index = {name: idx for idx, name in enumerate(sub_label_list)}

def get_file_label_pairs(folders, main_label):
    pairs = []
    for folder in folders:
        files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
        for f in files:
            sub_label = os.path.basename(folder)
            pairs.append((f, main_label, sub_label))
    return pairs

tumour_files = get_file_label_pairs(tumour_folders, main_label=1)
immune_files = get_file_label_pairs(immune_folders, main_label=0)
all_file_label_pairs = tumour_files + immune_files
print(f"Total images before balancing: {len(all_file_label_pairs)}")

def create_balanced_equal_dataset(file_label_pairs, target_size=15000):
    label_to_files = defaultdict(list)
    for f, main_label, sub_label in file_label_pairs:
        label_to_files[sub_label].append((f, main_label, sub_label))

    balanced = []
    for sub_label, files in label_to_files.items():
        originals = files.copy()
        current_size = len(originals)

        if current_size < target_size:
            extra_needed = target_size - current_size
            oversampled_files = random.choices(originals, k=extra_needed)
            balanced.extend((f, m, s, False) for f, m, s in originals)
            balanced.extend((f, m, s, True) for f, m, s in oversampled_files)
        elif current_size > target_size:
            undersampled_files = random.sample(originals, k=target_size)
            balanced.extend((f, m, s, False) for f, m, s in undersampled_files)
        else:
            balanced.extend((f, m, s, False) for f, m, s in originals)

    random.shuffle(balanced)
    return balanced

target_size = 15000
balanced_pairs = create_balanced_equal_dataset(all_file_label_pairs, target_size = target_size)
print(f"Total images each class balances to: {target_size}")
print(f"Total images after balancing: {len(balanced_pairs)}")

def augment_image(img):
    angles = [0, 90, 180, 270]
    angle = random.choice(angles)
    
    if angle == 90:
        img = tf.image.rot90(img, k=1)
    elif angle == 180:
        img = tf.image.rot90(img, k=2)
    elif angle == 270:
        img = tf.image.rot90(img, k=3)

    flip_type = random.choice(['none', 'horizontal', 'vertical', 'both'])
    if flip_type == 'horizontal':
        img = tf.image.flip_left_right(img)
    elif flip_type == 'vertical':
        img = tf.image.flip_up_down(img)
    elif flip_type == 'both':
        img = tf.image.flip_left_right(img)
        img = tf.image.flip_up_down(img)

    return img

def decode_img(img_path, augment=False):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_width, img_height])
    img = img / 255.0
    if augment:
        img = augment_image(img)
    return img

def prepare_dataset(pairs):
    file_paths, main_labels, sub_labels, augment_flags = [], [], [], []

    for f, m, s, a in pairs:
        file_paths.append(f)
        main_labels.append(m)
        sub_labels.append(sub_label_to_index[s])
        augment_flags.append(a)

    file_paths = np.array(file_paths)
    main_labels = np.array(main_labels)
    sub_labels = np.array(sub_labels)
    augment_flags = np.array(augment_flags)

    train_files, test_files, train_main, test_main, train_sub, test_sub, train_aug, test_aug = train_test_split(
        file_paths, main_labels, sub_labels, augment_flags, test_size=0.2, stratify=sub_labels, random_state=seed
    )

    def process_path(file_path, main_label, sub_label, augment):
        img = decode_img(file_path, augment)
        return img, {"binary_output": main_label, "multiclass_output": sub_label}

    train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_main, train_sub, train_aug))
    train_ds = train_ds.map(process_path, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_main, test_sub, test_aug))
    test_ds = test_ds.map(process_path, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    return train_ds, test_ds

def build_multitask_model():
    inputs = tf.keras.Input(shape=(img_width, img_height, 3))
    x = tf.keras.layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Conv2D(64, (3,3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)

    binary_output = tf.keras.layers.Dense(1, activation='sigmoid', name="binary_output")(x)
    multiclass_logits = tf.keras.layers.Dense(8, name="raw_multiclass_logits")(x)
    scaled_logits = tf.keras.layers.Multiply()([multiclass_logits, binary_output])
    multiclass_output = tf.keras.layers.Activation('softmax', name="multiclass_output")(scaled_logits)

    model = tf.keras.Model(inputs=inputs, outputs=[binary_output, multiclass_output])

    model.compile(
        optimizer='adam',
        loss={
            "binary_output": "binary_crossentropy",
            "multiclass_output": "sparse_categorical_crossentropy"
        },
        metrics={
            "binary_output": "accuracy",
            "multiclass_output": "accuracy"
        }
    )

    return model

def train_and_evaluate(train_ds, test_ds, title="Model"):
    model = build_multitask_model()
    model.summary()

    history = model.fit(train_ds, epochs=10, validation_data=test_ds)

    history_df = pd.DataFrame(history.history)

    results = model.evaluate(test_ds)
    print(f"\nTest results - {title}:")
    print(f"Binary Accuracy: {results[3]:.4f}")
    print(f"Multiclass Accuracy: {results[4]:.4f}")

    y_true, y_pred, confidences = [], [], []

    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        bin_preds = preds[0].flatten()
        multi_preds = preds[1]

        true_bin = labels['binary_output'].numpy().flatten()
        true_multi = labels['multiclass_output'].numpy().flatten()

        tumour_indices = np.where(true_bin == 1)[0]
        if len(tumour_indices) > 0:
            y_true.extend(true_multi[tumour_indices])
            y_pred.extend(np.argmax(multi_preds[tumour_indices], axis=1))
            confidences.extend(np.max(multi_preds[tumour_indices], axis=1))

    if y_true and y_pred:
        y_true_tumour = np.array(y_true)
        y_pred_tumour = np.array(y_pred)

    return model, y_true_tumour, y_pred_tumour, history_df

train_ds_over, test_ds_over = prepare_dataset(balanced_pairs)

print("\n--- Training with MS+Non-CLAHE ---")
model_over, y_true_tumour, y_pred_tumour, history_df = train_and_evaluate(train_ds_over, test_ds_over, title="MS+Non-CLAHE")

Total images before balancing: 124622
Total images each class balances to: 15000
Total images after balancing: 120000

--- Training with MS+Non-CLAHE ---


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)    │ (None, 100, 100, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 98, 98, 32)        │             896 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_4               │ (None, 49, 49, 32)        │               0 │ conv2d_4[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_5 (Conv2D)             │ (None, 47, 47, 64)        │          18,496 │ max_pooling2d_4[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_5               │ (None, 23, 23, 64)        │               0 │ conv2d_5[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_2 (Flatten)           │ (None, 33856)             │               0 │ max_pooling2d_5[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 128)               │       4,333,696 │ flatten_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ binary_output (Dense)         │ (None, 1)                 │             129 │ dense_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ raw_multiclass_logits (Dense) │ (None, 8)                 │           1,032 │ dense_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multiply_2 (Multiply)         │ (None, 8)                 │               0 │ raw_multiclass_logits[0][… │
│                               │                           │                 │ binary_output[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multiclass_output             │ (None, 8)                 │               0 │ multiply_2[0][0]           │
│ (Activation)                  │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,354,249 (16.61 MB)

 Trainable params: 4,354,249 (16.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 147s 761ms/step - binary_output_accuracy: 0.6897 - binary_output_loss: 0.6031 - loss: 2.5847 - multiclass_output_accuracy: 0.1971 - multiclass_output_loss: 1.9816 - val_binary_output_accuracy: 0.8388 - val_binary_output_loss: 0.3965 - val_loss: 2.1247 - val_multiclass_output_accuracy: 0.3166 - val_multiclass_output_loss: 1.7283
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 129s 682ms/step - binary_output_accuracy: 0.8477 - binary_output_loss: 0.3936 - loss: 2.0891 - multiclass_output_accuracy: 0.3288 - multiclass_output_loss: 1.6955 - val_binary_output_accuracy: 0.8589 - val_binary_output_loss: 0.3844 - val_loss: 1.9800 - val_multiclass_output_accuracy: 0.3568 - val_multiclass_output_loss: 1.5957
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 141s 677ms/step - binary_output_accuracy: 0.8809 - binary_output_loss: 0.3463 - loss: 1.9070 - multiclass_output_accuracy: 0.3750 - multiclass_output_loss: 1.5607 - val_binary_output_accuracy: 0.8856 - val_binary_output_

In [8]:
model_over.save("Image-10/Image10_app/models/dlcnn_MS_NONCLAHE.keras")
history_df.to_csv("Image-10/Image10_app/models/history/dlcnn_MS_NONCLAHE_history.csv", index=False)
np.save("Image-10/Image10_app/models/outputs/dlcnn_MS_NONCLAHE_ytrue.npy", y_true_tumour)
np.save("Image-10/Image10_app/models/outputs/dlcnn_MS_NONCLAHE_ypred.npy", y_pred_tumour)

all_confidences = []

for images, labels in test_ds_over:
    preds = model_over.predict(images)
    bin_preds = preds[0].flatten()
    multi_preds = preds[1]
    
    true_bin = labels['binary_output'].numpy().flatten()
    
    tumour_mask = (true_bin == 1) & (bin_preds > 0.5)
    
    filtered_multi_preds = multi_preds[tumour_mask]
    confidences = np.max(filtered_multi_preds, axis=1)
    all_confidences.extend(confidences)

np.save("Image-10/Image10_app/models/outputs/dlcnn_MS_NONCLAHE_conf.npy", np.array(all_confidences))

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
